# Validação do parser Carros na Web com fallback de LLM

Notebook para testar o parser genérico com `use_llm_fallback=True` nos PDFs do Corolla Cross e do Taos.
A comparação final é feita contra as listas curadas em `corolla_cross_rows.py` e `taos_rows.py`.

In [1]:
from pathlib import Path

import pandas as pd

from carros_na_web_parser import parse_pdf, parse_pdf_with_sections
from corolla_cross_rows import get_feature_rows as get_corolla_rows
from taos_rows import get_feature_rows as get_taos_rows

In [2]:
pdfs = {
    'corolla_cross': {
        'pdf': Path('corolla_cross_source.pdf'),
        'expected_rows': get_corolla_rows(),
    },
    'taos': {
        'pdf': Path('taos_source.pdf'),
        'expected_rows': get_taos_rows(),
    },
}

pdfs

{'corolla_cross': {'pdf': WindowsPath('corolla_cross_source.pdf'),
  'expected_rows': [('Marca', 'Toyota'),
   ('Modelo', 'Corolla Cross XRX 2.0'),
   ('Ano', '2026'),
   ('Preço', 'R$ 194.700'),
   ('Desvalorização', '1,46%'),
   ('Propulsão', 'Combustão'),
   ('Combustível', 'Flex (álcool/gasolina)'),
   ('IPVA', 'R$ 7.788'),
   ('Seguro', 'R$ 6.230'),
   ('Revisões', 'R$ 4.782 até 50.000'),
   ('Procedência', 'Nacional'),
   ('Garantia', '10 anos'),
   ('Configuração', 'SUV'),
   ('Porte', 'Médio'),
   ('Lugares', '5'),
   ('Portas', '4'),
   ('Plataforma', 'TNGA'),
   ('Nota do leitor', '8,7'),
   ('Índice CNW', '5.414,96'),
   ('Ranking CNW', '809'),
   ('Instalação', 'Dianteiro'),
   ('Aspiração', 'Natural'),
   ('Disposição', 'Transversal'),
   ('Alimentação', 'Injeção direta e indireta'),
   ('Cilindros', '4 em linha'),
   ('Comando de válvulas', 'Duplo no cabeçote'),
   ('Tuchos', 'Hidráulicos'),
   ('Variação do comando', 'Admissão e escape'),
   ('Cilindrada unitária', '497 

In [4]:
def as_reference_dataframe(rows):
    return pd.DataFrame(rows, columns=['Atributo veicular', 'Valor'])


def compare_frames(actual: pd.DataFrame, expected: pd.DataFrame):
    actual_pairs = [tuple(x) for x in actual[['Atributo veicular', 'Valor']].to_numpy()]
    expected_pairs = [tuple(x) for x in expected[['Atributo veicular', 'Valor']].to_numpy()]
    actual_set = set(actual_pairs)
    expected_set = set(expected_pairs)

    missing = [pair for pair in expected_pairs if pair not in actual_set]
    extra = [pair for pair in actual_pairs if pair not in expected_set]

    return {
        'rows_actual': len(actual),
        'rows_expected': len(expected),
        'missing_count': len(missing),
        'extra_count': len(extra),
        'missing': missing,
        'extra': extra,
    }


In [12]:
results = {}

for name, info in pdfs.items():
    pdf_path = info['pdf']
    expected_df = as_reference_dataframe(info['expected_rows'])
    actual_df = parse_pdf(pdf_path, use_llm_fallback=True)
    summary = compare_frames(actual_df, expected_df)
    results[name] = {
        'expected': expected_df,
        'actual': actual_df,
        'summary': summary,
    }

    print(f'=== {name} ===')
    print(f"rows expected: {summary['rows_expected']}")
    print(f"rows actual:   {summary['rows_actual']}")
    print(f"missing:       {summary['missing_count']}")
    print(f"extra:         {summary['extra_count']}")
    print('\nActual head:')
    print(actual_df.head(12).to_string(index=False))
    print('\nActual tail:')
    print(actual_df.tail(12).to_string(index=False))
    print('\n---\n')

=== corolla_cross ===
rows expected: 178
rows actual:   158
missing:       70
extra:         50

Actual head:
Atributo veicular               Valor
              Ano                2026
            Preço          R$ 194.700
   Desvalorização               1,46%
        Propulsão           Combustão
             IPVA            R$ 7.788
           Seguro            R$ 6.230
         Revisões R$ 4.782 até 50.000
      Procedência            Nacional
         Garantia             10 anos
            Porte               Médio
          Lugares                   5
           Portas                   4

Actual tail:
                           Atributo veicular    Valor
                   Tampa traseira motorizada Presente
                  Alças de segurança no teto Presente
                                       Rádio Presente
                          Tomada de 12 volts Presente
                      Volante multifuncional Presente
                Termômetro do líquido de arr Presente
    

In [10]:
results['corolla_cross']['actual'].to_csv('corolla_cross_actual.csv', index=False)

In [13]:
for name, data in results.items():
    summary = data['summary']
    print(f'=== {name} diffs ===')
    if summary['missing']:
        print('Missing rows:')
        for row in summary['missing'][:25]:
            print(row)
    else:
        print('No missing rows.')

    if summary['extra']:
        print('\nExtra rows:')
        for row in summary['extra'][:25]:
            print(row)
    else:
        print('No extra rows.')
    print('\n---\n')

=== corolla_cross diffs ===
Missing rows:
('Marca', 'Toyota')
('Modelo', 'Corolla Cross XRX 2.0')
('Combustível', 'Flex (álcool/gasolina)')
('Configuração', 'SUV')
('Nota do leitor', '8,7')
('Índice CNW', '5.414,96')
('Ranking CNW', '809')
('Alimentação', 'Injeção direta e indireta')
('Comando de válvulas', 'Duplo no cabeçote')
('Razão de compressão', '13:1')
('Deslocamento', '1987 cm3')
('Potência máxima', '167 cv (A) / 177 cv (G)')
('Código do motor', 'M20A-FKB')
('Torque máximo', '21,3 kgfm (A) / 20,8 kgfm (G)')
('Peso/potência', '8,11 kg/cv')
('Torque específico', '10,7 kgfm/litro')
('Peso/torque', '66,7 kg/kgfm')
('Potência específica', '88,1 cv/litro')
('Viscosidade do óleo', '0W-20 ou 5W-30')
('Câmbio', 'CVT de 10 marchas')
('Suspensão dianteira', 'Independente, McPherson')
('Elemento elástico dianteiro', 'Mola helicoidal')
('Suspensão traseira', 'Eixo de torção')
('Elemento elástico traseiro', 'Mola helicoidal')
('Freios dianteiros', 'Disco ventilado')

Extra rows:
('Índice', '

In [14]:
# Versão com seções para inspecionar o que o LLM devolveu em cada bloco.
for name, info in pdfs.items():
    df_sections = parse_pdf_with_sections(info['pdf'], use_llm_fallback=True)
    print(f'=== {name} with sections ===')
    print(df_sections.head(15).to_string(index=False))
    print(df_sections.tail(15).to_string(index=False))
    print('\n---\n')

=== corolla_cross with sections ===
section subsection Atributo veicular               Valor
    NaN        NaN               Ano                2026
    NaN        NaN             Preço          R$ 194.700
    NaN        NaN    Desvalorização               1,46%
    NaN        NaN         Propulsão           Combustão
    NaN        NaN              IPVA            R$ 7.788
    NaN        NaN            Seguro            R$ 6.230
    NaN        NaN          Revisões R$ 4.782 até 50.000
    NaN        NaN       Procedência            Nacional
    NaN        NaN          Garantia             10 anos
    NaN        NaN             Porte               Médio
    NaN        NaN           Lugares                   5
    NaN        NaN            Portas                   4
    NaN        NaN        Plataforma                TNGA
    NaN     Avalie            Índice                 CNW
    NaN     Avalie           Ranking                 CNW
      section subsection                            

In [11]:
results2 = {}

for name, info in pdfs.items():
    pdf_path = info['pdf']
    expected_df = as_reference_dataframe(info['expected_rows'])
    actual_df = parse_pdf(pdf_path, use_llm_fallback=False)
    summary = compare_frames(actual_df, expected_df)
    results2[name] = {
        'expected': expected_df,
        'actual': actual_df,
        'summary': summary,
    }

    print(f'=== {name} ===')
    print(f"rows expected: {summary['rows_expected']}")
    print(f"rows actual:   {summary['rows_actual']}")
    print(f"missing:       {summary['missing_count']}")
    print(f"extra:         {summary['extra_count']}")
    print('\nActual head:')
    print(actual_df.head(12).to_string(index=False))
    print('\nActual tail:')
    print(actual_df.tail(12).to_string(index=False))
    print('\n---\n')

=== corolla_cross ===
rows expected: 178
rows actual:   158
missing:       75
extra:         55

Actual head:
Atributo veicular               Valor
              Ano                2026
            Preço          R$ 194.700
   Desvalorização               1,46%
        Propulsão           Combustão
             IPVA            R$ 7.788
           Seguro            R$ 6.230
         Revisões R$ 4.782 até 50.000
      Procedência            Nacional
         Garantia             10 anos
            Porte               Médio
          Lugares                   5
           Portas                   4

Actual tail:
                           Atributo veicular    Valor
                   Tampa traseira motorizada Presente
                  Alças de segurança no teto Presente
                                       Rádio Presente
                          Tomada de 12 volts Presente
                      Volante multifuncional Presente
                Termômetro do líquido de arr Presente
    